In [ ]:
from modeltool.model_building import iter_random_search
import hyperopt
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle("D:/Creditnow/02_Model/Creditnow_acq_sms_app_02/ModelDataMaid/3_data/creditnow_acq_sms_app_02_data.pkl")
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.replace([np.inf, -np.inf], 0)

In [3]:
df.shape

(4667, 28178)

#### Separate data into DEV and OOT

In [4]:
df_dev = df[df['order_date']>'2025-01-15']
df_oot = df[(df['order_date']<='2025-01-15')&(df['order_date']>='2025-01-01')]
print(df_dev.shape,df_oot.shape)

(3611, 28178) (920, 28178)


#### Declare search space

In [5]:
params_space = {     
        # 'type': 'xgb',
        'n_estimators': hyperopt.hp.quniform('n_estimators',15,30,1),
        'max_depth': hyperopt.hp.quniform('max_depth', 3, 8, 1),
        'learning_rate': hyperopt.hp.uniform('learning_rate', 0.001, 0.1),
        'min_child_weight': hyperopt.hp.quniform('min_child_weight', 1, 10, 1),
        'gamma': hyperopt.hp.uniform('gamma', 1, 10),
        'subsample': hyperopt.hp.uniform('subsample', 0.4, 0.8),
        'colsample_bytree': hyperopt.hp.uniform('colsample_bytree', 0.5, 0.9),
        'reg_alpha': hyperopt.hp.uniform('reg_alpha', 0, 5),
        'reg_lambda': hyperopt.hp.uniform('reg_lambda', 0, 5),
        'scale_pos_weight': hyperopt.hp.uniform('scale_pos_weight', 0.3, 0.5),
    } 

#### If you have a varlist to start with

In [6]:
varlist = pd.read_csv("D:/Creditnow/02_Model/Creditnow_acq_sms_app_02/var/round_18071_61.csv")
varlist = varlist[varlist['imp']>0]['col'].to_list()
len(varlist)

59

#### Start searching

In [ ]:
iter_random_search(df_dev=df_dev,
                    df_oot={'oot':df_oot},
                    dep='t0_cnt',
                    param_space=params_space,
                    ntrials=60,
                    varlist=varlist,
                    generate_report=True
                    )

  3%|▎         | 2/60 [00:00<00:06,  8.37trial/s, best loss: 0.4023972602739726]

d:\Model inventory\ModelTool\modeltool\random_search.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_oot.loc[:,'pred'] = xgb_model.predict_proba(df_oot[varlist])[:,1]



100%|██████████| 60/60 [00:07<00:00,  7.89trial/s, best loss: 0.38898040575689263]
round_59: {'oot': [0.6056281953247935, 0.1870019401032615], 'dev': [0.7559797310472707, 0.3792513569952173], 'test': [0.6110195942431074, 0.19394832668631873]}
round_59: {'colsample_bytree': 0.8253427427893854, 'gamma': 4.060225224859025, 'learning_rate': 0.05545740611202673, 'max_depth': 7, 'min_child_weight': 8, 'n_estimators': 22, 'reg_alpha': 3.641794065594113, 'reg_lambda': 2.3468195552430138, 'scale_pos_weight': 0.40669131569622263, 'subsample': 0.7778071515542755, 'importance_type': 'total_gain', 'eval_metric': 'auc'}
100%|██████████| 60/60 [00:06<00:00,  9.44trial/s, best loss: 0.3900569331252528] 
round_49: {'oot': [0.6020829251954323, 0.19289176498840366], 'dev': [0.7427049319818335, 0.3613622423836316], 'test': [0.6099430668747472, 0.1744263337379342]}
round_49: {'colsample_bytree': 0.645506555726196, 'gamma': 3.103228072561758, 'learning_rate': 0.05543779158513397, 'max_depth': 4, 'min_child_

Running the function above generate 3 folders: [model/report/var] the models on the road will all be saved at corresponding folders